In [3]:
import sys, os, time, pprint
import numpy as np

# Import custom functions for splitting and search.
sys.path.append("..")  # Adds higher directory to python modules path.
from shared import milvus_utilities as _utils

In [7]:
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

In [3]:
from pymilvus import connections, utility
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file
TOKEN = os.environ["ZILLIZ_API_KEY"]

CLUSTER_ENDPOINT="https://in03-5223ff782a72af1.serverless.aws-eu-central-1.cloud.zilliz.com"
connections.connect(
  alias='default',
  #  Public endpoint obtained from Zilliz Cloud
  uri=CLUSTER_ENDPOINT,
  # API key or a colon-separated cluster username and password
  token=TOKEN,
)

# Check if the server is ready and get colleciton name.
print(f"Type of server: {utility.get_server_version()}")

Type of server: Zilliz Cloud Vector Database(Compatible with Milvus 2.6)


In [4]:
import torch
from torch.nn import functional as F
from sentence_transformers import SentenceTransformer

# Initialize torch settings
torch.backends.cudnn.deterministic = True
DEVICE = torch.device('cuda:3' if torch.cuda.is_available() else 'cpu')
print(f"device: {DEVICE}")


model_name = "WhereIsAI/UAE-Large-V1"
encoder = SentenceTransformer(model_name, device=DEVICE)
print(type(encoder))
print(encoder)

# Get the model parameters and save for later.
EMBEDDING_DIM = encoder.get_sentence_embedding_dimension()
MAX_SEQ_LENGTH_IN_TOKENS = encoder.get_max_seq_length() 

MAX_SEQ_LENGTH = MAX_SEQ_LENGTH_IN_TOKENS
HF_EOS_TOKEN_LENGTH = 1

# Inspect model parameters.
print(f"model_name: {model_name}")
print(f"EMBEDDING_DIM: {EMBEDDING_DIM}")
print(f"MAX_SEQ_LENGTH: {MAX_SEQ_LENGTH}")

/Users/joeljvarghese/Documents/Workspace/Milvus_ollama_trial/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cpu
<class 'sentence_transformers.SentenceTransformer.SentenceTransformer'>
SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)
model_name: WhereIsAI/UAE-Large-V1
EMBEDDING_DIM: 1024
MAX_SEQ_LENGTH: 512


In [5]:
from pymilvus import MilvusClient

COLLECTION_NAME = "wikipedia"

M = 16
efConstruction = M * 2
INDEX_PARAMS = dict({
    'M': M,               
    "efConstruction": efConstruction })
index_params = {
    "index_type": "HNSW", 
    "metric_type": "COSINE", 
    "params": INDEX_PARAMS
    }

mc = MilvusClient(
    uri=CLUSTER_ENDPOINT,
    # API key or a colon-separated cluster username and password
    token=TOKEN)

# Check if collection already exists, if so drop it.
has = utility.has_collection(COLLECTION_NAME)
if has:
    drop_result = utility.drop_collection(COLLECTION_NAME)
    print(f"Successfully dropped collection: `{COLLECTION_NAME}`")

has = utility.has_collection(COLLECTION_NAME)
if has:
    drop_result = utility.drop_collection(COLLECTION_NAME)
    print(f"Successfully dropped collection: `{COLLECTION_NAME}`")

# Create the collection.
mc.create_collection(COLLECTION_NAME, 
                     EMBEDDING_DIM,
                     consistency_level="Eventually", 
                     auto_id=True,
                     # skip setting params below, if using AUTOINDEX
                     params=index_params
                    )

print(f"Successfully created collection: `{COLLECTION_NAME}`")

Successfully dropped collection: `wikipedia`
Successfully created collection: `wikipedia`


In [8]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# load the Wikipedia page and create index
loader = WebBaseLoader("https://en.wikipedia.org/wiki/New_York_City")
docs = loader.load()

# Split the documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50)
print(f"Num docs: {len(docs)}")
chunks = text_splitter.split_documents(docs)
print(f"Num chunks: {len(chunks)}")

# Convert chunks to a list of dictionaries.
chunk_list = []
for chunk in chunks:
    # pprint.pprint(chunk)
    # Generate embeddings using encoder from HuggingFace.
    embeddings = torch.tensor(encoder.encode([chunk.page_content]))
    embeddings = np.array(embeddings / np.linalg.norm(embeddings)) #use numpy
    converted_values = list(map(np.float32, embeddings))[0]
    
    # Assemble embedding vector, original text chunk, metadata.
    chunk_dict = {
        'vector': converted_values,
        'chunk': chunk.page_content,
        'source': chunk.metadata['source'],
        'h1': chunk.metadata['title'][:50],
    }
    chunk_list.append(chunk_dict)

# Insert data into the Milvus collection.
print("Start inserting entities")
start_time = time.time()
insert_result = mc.insert(
    COLLECTION_NAME,
    data=chunk_list,
    append=True,
    progress_bar=True)
end_time = time.time()
print(f"Milvus Client insert time for {len(chunk_list)} vectors: {end_time - start_time} seconds")
# Milvus Client insert time for 646 vectors: 4.732278823852539 seconds

# After final entity is inserted, call flush to stop growing segments left in memory.
mc.flush(COLLECTION_NAME)

Num docs: 1
Num chunks: 761


/var/folders/r4/dp3q02rx14lcf4mg7lgqtnp80000gn/T/ipykernel_72468/2821819526.py:20: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  embeddings = np.array(embeddings / np.linalg.norm(embeddings)) #use numpy


Start inserting entities
Milvus Client insert time for 761 vectors: 2.277195692062378 seconds


In [8]:
from langchain_groq import ChatGroq

api = os.getenv("GROQ_API_KEY")

LLM_NAME = "openai/gpt-oss-120b"
RANDOM_SEED = 415

# Reasonable values for the penalty coefficients are around 0.1 to 1 if the aim is to just reduce repition 
# somewhat. To strongly suppress repetition, set coefficients = 2.
FREQUENCY_PENALTY = 2

# See how to save api key in env variable.
# https://help.openai.com/en/articles/5112595-best-practices-for-api-key-safety
groq_client = ChatGroq(
    api_key = api,
    model = LLM_NAME,
    temperature=0.1
)

In [13]:
from datasets import Dataset
from ragas import evaluate

from ragas.metrics import (
    context_recall, 
    context_precision, 
    # Context -> Answer metrics
    faithfulness, 
    # Question -> Answer metrics
    answer_similarity,
    answer_relevancy, 
    answer_correctness
    )

metric_objects = [
    context_recall,
    context_precision,
    answer_relevancy,
    faithfulness,
    answer_similarity,
    answer_correctness,
]

llm_langchain = ChatGroq(
    api_key = api,
    model = LLM_NAME,
    temperature=0.1
)


for metric in metric_objects:
    metric.llm = llm_langchain




/var/folders/r4/dp3q02rx14lcf4mg7lgqtnp80000gn/T/ipykernel_73754/2667367012.py:4: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import (
/var/folders/r4/dp3q02rx14lcf4mg7lgqtnp80000gn/T/ipykernel_73754/2667367012.py:4: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
/var/folders/r4/dp3q02rx14lcf4mg7lgqtnp80000gn/T/ipykernel_73754/2667367012.py:4: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/var/

In [14]:
def assemble_ragas_dataset(input_df, answer_col_name="OpenAI_RAG_answer", context_exists=False, row_number=-9999):
    """Assemble a RAGAS HuggingFace Dataset from lists of values."""

    # Subset input_df to the row number.
    if row_number >= 0:
        subset_df = input_df.iloc[row_number:row_number+1, :]
    else:
        subset_df = input_df.copy()

    question_list = subset_df.Question.to_list()
    answer_list = subset_df[answer_col_name].to_list()

    # contexts: list[list[str]] - The contexts which were passed into the LLM to answer the question.
    if context_exists:
        context_list = subset_df.Custom_RAG_context.to_list()
        context_list = [[context] for context in context_list]
    else:
        context_list = [ [""] for _ in question_list]

    # ground_truths: list[list[str]] - The ground truth answer to the questions. 
    truth_list = subset_df.ground_truth_answer.to_list()
    truth_list = [[truth] for truth in truth_list]

    # Create a HuggingFace Dataset from the ground truth lists.
    ragas_ds = Dataset.from_dict({"question": question_list,
                            "contexts": context_list,
                            "answer": answer_list,
                            "ground_truths": truth_list})
    
    return ragas_ds

def evaluate_ragas(input_df, answer_col_name="OpenAI_RAG_answer", context_exists=False, row_number=-9999, metrics="final_only"):

    # Create a ragas dataset.
    ragas_input_ds = assemble_ragas_dataset(input_df, answer_col_name, context_exists, row_number)

    # Evaluate the dataset.
    if metrics == "final_only":
        ragas_result = evaluate(
            ragas_input_ds,
            metrics=[
                answer_similarity,
                answer_relevancy,
                answer_correctness,])
    else:
        # calculate all metrics
        ragas_result = evaluate(
            ragas_input_ds,
            metrics=[
                # Question -> Context metrics
                context_recall, 
                context_precision, 
                # Context -> Answer metrics
                faithfulness, 
                # Question -> Answer metrics
                answer_similarity,
                answer_relevancy,
                answer_correctness,])
        
    return ragas_result


In [ ]:
import pandas as pd

# Read ground truth answers from file.
eval_df = pd.read_csv("../../../christy_coding_scratch/data/milvus_ground_truth.csv", 
                      header=0, skip_blank_lines=True)
display(eval_df.head())

# Get all the questions.
question_list = eval_df.Question.to_list()

# Get all the ground truth answers.
truth_list = eval_df.ground_truth_answer.to_list()

# Get all the ground truth sources.
uri_list = eval_df.Uri.to_list()

# Get all the OpenAI Answers.
openai_answer_list = eval_df.OpenAI_RAG_answer.to_list()

In [ ]:
import requests, json, pprint

# Milvus search, define how many retrieval results to return.
# Milvus automatically sorts results descending by distance score.
TOP_K = 3

# Search a collection containing Milvus Documentation.
def zilliz_pipeline_collection_search(token, question):
    # Define the URL, headers, and data
    url = "https://controller.api.gcp-us-west1.zillizcloud.com/v1/pipelines/pipe-3de3fb4a9bc3c2a64a786b/run"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}",
    }
    data = {
        "data": {
            "query_text": question
        },
        "params": {
            "limit": 3,
            "offset": 0,
            "outputFields": ["chunk_text", "chunk_id", "doc_name", "source"],
            "filter": "chunk_id >= 0 && doc_name == 'param.html'",
        }
    }

    # Send the POST request
    response = requests.post(url, headers=headers, json=data)

    # # Print the response
    # pprint.pprint(response.json())
    return response.json()

# Search a collection containing Wikipedia articles about New York City.
def wikipedia_search(mc, collection_name, collection_encoder, question, output_fields=None, top_k=3):
    # Embed the query
    query_embeddings = _utils.embed_query(collection_encoder, [question])

    # Define search parameters
    INDEX_PARAMS = dict({
        'M': M,               
        "efConstruction": efConstruction })
    SEARCH_PARAMS = dict({
        "ef": INDEX_PARAMS['efConstruction']
    })

    # Define output fields to return
    OUTPUT_FIELDS = ["h1", "source", "chunk"]

    # Perform the search
    answers = mc.search(
        collection_name,
        data=query_embeddings, 
        search_params=SEARCH_PARAMS,
        output_fields=output_fields, 
        filter="(source like 'https://en.wikipedia.org%')",
        limit=top_k,
        consistency_level="Eventually"
    )

    return answers

In [ ]:
# Function to get OpenAI response and token usage.
def get_openai_chat(llm_name, user_prompt, retrieval_context, retrieval_source, message_history,
                     temperature=0.0, random_seed=415, frequency_penalty=2):
    """ 
    Returns 2 pandas dataframes: response, token_use.
    """
    
    system_message = f"""
    Use the Context to answer the user's question. Be clear, factual, complete, concise.
    If the answer is not in the Context, say "I don't know".  Otherwise answer using this format:
    Context: {retrieval_context}
    Answer: The answer to the question.
    Grounding source: {retrieval_source}
    """
    messages = [
        {'role': 'system', 'content': system_message},
        {'role': 'user', 'content': f"{user_prompt}"},
        {'role': 'assistant', 'content': f"Relevant context:\n{retrieval_context}"}
    ]

    # Define the OpenAIEvaluator.
    responses = groq_client.chat.completions.create(
        response_format={
            "type": "json_object", 
            # "schema": Result.schema_json()
        },
        messages=message_history + messages,
        model=llm_name,
        temperature=temperature, # the degree of randomness of the model's output
        seed=random_seed,  # for reproducibility
        frequency_penalty=frequency_penalty, # allowed amount of repitition in the model's output
        # max_tokens=max_tokens # maximum number of tokens the model can output
    )
    message_history = message_history + messages[1:]

    # Make sure total_tokens < 4096.
    token_dict = {
        'prompt_tokens':responses.usage.prompt_tokens,
        'completion_tokens':responses.usage.completion_tokens,
        'total_tokens':responses.usage.total_tokens,
    }

    # Return answer as a JSON object.
    openai_response = responses.choices[0].message.content
    json_response = json.loads(openai_response)
    json_response # single json object with 3 fields

    # Create a DataFrame from a list of dictionaries.
    response_df = pd.DataFrame([json_response])
    token_use_df = pd.DataFrame([token_dict])

    return response_df, token_use_df

def get_answer_from_openai_chat_response(chat_response):
    # Extract the answer from the 0th choice's message content
    answer = chat_response.choices[0].message.content
    return answer